# Frozen-checkpoint tie-breaking audit

Select a T4 GPU, then run cells in order. The notebook embeds the evaluator source. Upload the original RotatE ZIP and the corrected ConvKB ZIP (20261111); hashes are checked before loading. The GPU workflow is prepared but has not been executed in the local CPU environment.


In [ ]:
import base64, io, os, zipfile
from pathlib import Path
PROJECT = Path('/content/tie_breaking_audit')
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:
    z.extractall(PROJECT)
os.chdir(PROJECT)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-evaluation.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime and reconnect.'
print(torch.__version__, torch.cuda.get_device_name(0))


Google Drive is optional. The default saves results on the Colab VM; download the final bundle before the runtime is deleted. Set USE_GOOGLE_DRIVE=True for persistent storage. A failed mount falls back explicitly to a local run; it does not resume inaccessible Drive results. Keep the same packages, GPU, source, batch and chunk sizes when resuming.


In [ ]:
from pathlib import Path
from google.colab import files
USE_GOOGLE_DRIVE = False
OUTPUT = Path('/content/TieBreakingAudit/final_audit')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        OUTPUT = Path('/content/drive/MyDrive/TieBreakingAudit/final_audit')
    except (ValueError, OSError, RuntimeError) as error:
        print(f'Drive mount failed: {error}')
        print('Using local storage; inaccessible Drive progress will not be resumed.')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT)
if str(OUTPUT).startswith('/content/TieBreakingAudit/'):
    print('Local storage: download the final bundle before deleting the runtime.')


In [ ]:
uploaded = files.upload()
import hashlib
hash_to_path = {hashlib.sha256(data).hexdigest(): str(Path(name).resolve()) for name, data in uploaded.items()}
from audit_pipeline import ARCHIVES
missing = [model for model, checksum in ARCHIVES.items() if checksum not in hash_to_path]
if missing:
    raise ValueError('Missing or incorrect ZIP packages: ' + ', '.join(missing))
rotate_zip = hash_to_path[ARCHIVES['rotate']]
convkb_zip = hash_to_path[ARCHIVES['convkb']]


In [ ]:
command = [sys.executable, '-u', 'audit_pipeline.py', '--rotate-archive', rotate_zip,
           '--convkb-archive', convkb_zip, '--out', str(OUTPUT), '--device', 'cuda', '--threads', '4']
if (OUTPUT / 'rotate_pilot' / 'metadata.json').exists():
    command.append('--resume')
subprocess.run(command, check=True)


The runner verifies both packages, runs all tests, compares validation examples, pilots both models, then scores 40,932 test queries per model. Model progress is saved in `rotate_test.log` and `convkb_test.log`. Re-run the preceding cell to resume with matching settings.


In [ ]:
bundle = OUTPUT.parent / 'evaluation_validation_bundle.zip'
files.download(str(bundle))
files.download(str(bundle.with_suffix('.zip.sha256')))
print((OUTPUT / 'main_table.md').read_text())
